# run06 H: fork of dddmdd ModernBERT+UltraBERT+Ettin (LB 0.568)

Inference-only 3-model x 5-fold BERT-family blend, equal weights. Adds blend-logit dump (forkH_probs.npz) + canonical label-order file (probs_order.npy) for run06 offline blends.
Original notebook: dddmdd/modernbert-ultrabert-ettin-lb-0-568 (public, LB 0.568).


In [ ]:
import glob, json, os, time
from concurrent.futures import ThreadPoolExecutor
import numpy as np, pandas as pd, torch
import transformers
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("transformers", transformers.__version__, "| torch", torch.__version__, "| GPUs", torch.cuda.device_count(), flush=True)
CFG0 = sorted(glob.glob("/kaggle/input/**/modernbert-large/fold0/ntl_config.json", recursive=True))[0]
ROOT = os.path.dirname(os.path.dirname(os.path.dirname(CFG0)))   # <dataset>/modernbert-large/fold0/ntl_config.json
TEST = (glob.glob("/kaggle/input/competitions/name-that-llm/test.csv") + glob.glob("/kaggle/input/name-that-llm/test.csv"))[0]
WEIGHTS = {"modernbert-large": 1.0, "ultrabert-380m": 1.0, "ettin-encoder-400m": 1.0}
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
BATCH = 32
test = pd.read_csv(TEST)
print("dataset", ROOT, "| test", test.shape, flush=True)


@torch.no_grad()
def predict_fold(fold_dir, texts):
    """Softmax probabilities of one fold checkpoint (passages x labels)."""
    cfg = json.load(open(os.path.join(fold_dir, "ntl_config.json")))
    trc = cfg["trust_remote_code"]
    tok = AutoTokenizer.from_pretrained(fold_dir, trust_remote_code=trc)
    dtype = torch.float16 if DEVICE == "cuda" else torch.float32
    model = AutoModelForSequenceClassification.from_pretrained(fold_dir, trust_remote_code=trc, dtype=dtype).to(DEVICE).eval()
    ids = tok(list(texts), truncation=True, max_length=cfg["max_len"])["input_ids"]
    order = np.argsort([len(x) for x in ids])   # sort by length to reduce padding
    probs = np.zeros((len(texts), len(cfg["labels"])), np.float32)
    for i in range(0, len(order), BATCH):
        idx = order[i:i + BATCH]
        batch = tok.pad({"input_ids": [ids[j] for j in idx]}, return_tensors="pt").to(DEVICE)
        probs[idx] = torch.softmax(model(**batch).logits.float(), -1).cpu().numpy()
    del model
    if DEVICE == "cuda":
        torch.cuda.empty_cache()
    return probs, cfg["labels"]


def read_file(path, block=16 << 20):
    with open(path, "rb") as f:
        while f.read(block):
            pass


# Read all weights first, with 8 threads. Kaggle datasets are much faster to read in parallel
# (measured: 26 MiB/s one file at a time, 132 MiB/s with 8 threads). Loading then comes from the page cache.
t0 = time.time()
FOLDS = {name: sorted(glob.glob(os.path.join(ROOT, name, "fold*"))) for name in WEIGHTS}
pool = ThreadPoolExecutor(8)
prefetch = {d: pool.submit(read_file, os.path.join(d, "model.safetensors")) for name in WEIGHTS for d in FOLDS[name]}

labels, model_logp = None, {}
for name in WEIGHTS:
    folds = FOLDS[name]
    assert len(folds) == 5, (name, folds)
    p = 0
    for d in folds:
        prefetch[d].result()
        pf, lab = predict_fold(d, test.text)
        assert labels is None or lab == labels, "label order differs between models"
        labels = lab
        p = p + pf / len(folds)
        print(f"{name} {os.path.basename(d)} done ({time.time() - t0:.0f}s)", flush=True)
    model_logp[name] = np.log(np.clip(p, 1e-9, 1.0))

blend = sum(WEIGHTS[n] * model_logp[n] for n in WEIGHTS)
sub = pd.DataFrame({"id": test["id"], "model": np.array(labels)[blend.argmax(1)]})
sub.to_parquet("submission.parquet", index=False)
print(sub.head(), sub.model.value_counts(), f"total {time.time() - t0:.0f}s", sep="\n")

# --- run06 fork H: dump blend logits + label order for offline blends ---
CANON = ["gpt-5.5","gpt-5.6-luna","gpt-5.6-sol","gpt-5.6-terra","gpt-6-astra",
         "gemini-3.1-pro-preview","gemini-3.6-flash","gemini-3.7-flash","gemini-3.8-flash",
         "claude-opus-4-6","claude-opus-4-8","claude-opus-5","claude-sonnet-5","claude-fable-5",
         "qwen3.8-max","glm-5.3","glm-5.3-flash","kimi-k3"]
assert sorted(labels) == sorted(CANON), (labels, CANON)
order = np.array([labels.index(l) for l in CANON], dtype=np.int32)
np.savez("forkH_probs.npz", logit=np.asarray(blend, dtype=np.float32), labels=np.array(labels))
np.save("probs_order.npy", order)
print("dumped forkH_probs.npz + probs_order.npy", flush=True)
